In [ ]:
# Cell 1: 抑制警告 (环境配置)
# 目的: 保持输出整洁，屏蔽不必要的库升级提示
# 输入: None
# 输出: 系统状态初始化
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
# Cell 2: 初始化与数据加载
# 目的: 加载 Phase 2 清洗后的主数据，并配置分析的基础绘图主题
# 输入: data/cs2_pro_2026_Active_Master.csv
# 输出: 全局 DataFrame (df)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/cs2_pro_2026_Active_Master.csv')
plt.style.use('dark_background')
sns.set_theme(style='darkgrid', rc={'axes.facecolor': '#121212', 'figure.facecolor': '#121212'})
print(f'✅ 数据加载完成: 共计 {len(df)} 名选手')

In [ ]:
# Cell 3: 深度数据质检 (质量控制)
# 目的: 执行 3 级质检过滤。1. 移除关键字段缺失 2. 移除无效队名占位符 3. 全局不区分大小写去重
# 输入: 加载的原始 df
# 输出: data/cs2_pro_2026_Active_Cleaned.csv
original_len = len(df)

# 1. 移除关键字段缺失 (eDPI/分辨率)
df_clean = df.dropna(subset=['eDPI', 'Resolution']).copy()

# 2. 移除虚假选手 (选手名与战队名相同的情况通常是爬虫占位符)
df_clean = df_clean[df_clean['Player'].str.lower() != df_clean['Team'].str.lower()]

# 3. 全局不区分大小写去重 (确保每个选手仅有一个样本)
df_clean['Player_lower'] = df_clean['Player'].str.lower()
df_clean = df_clean.drop_duplicates(subset=['Player_lower'], keep='first')
df_clean = df_clean.drop(columns=['Player_lower'])

print(f'🛡️  数据质检完成')
print(f'📉 原始: {original_len} → 📈 提纯后: {len(df_clean)} 行')
df_clean.to_csv('data/cs2_pro_2026_Active_Cleaned.csv', index=False, encoding='utf-8-sig')

In [ ]:
# Cell 4: 特征工程补丁 (RAW 数据回填)
# 目的: 从 RAW 原始日志中提取 02 阶段未考虑的高级字段 (如亮度、Reflex、V-Sync 等)
# 输入: cs2_pro_2026_Active_Cleaned.csv & cs2_pro_detailed_RAW.csv
# 输出: data/cs2_pro_2026_Active_Final.csv
df_clean = pd.read_csv('data/cs2_pro_2026_Active_Cleaned.csv')
df_raw = pd.read_csv('data/cs2_pro_detailed_RAW.csv', low_memory=False)

# 指定需要同步的字段
target_columns = ['Player', 'Brightness', 'Display Mode', 'V-Sync', 'NVIDIA Reflex Low Latency', 
                  'NVIDIA G-Sync', 'Maximum FPS In Game', 'Model / Texture Detail', 
                  'Shader Detail', 'Particle Detail', 'Ambient Occlusion']

# 提取并合并
df_features = df_raw[target_columns].drop_duplicates(subset=['Player'])
df_pro = pd.merge(df_clean, df_features, on='Player', how='left', suffixes=('', '_new'))

# 字段覆盖逻辑
for col in target_columns:
    if col != 'Player' and col + '_new' in df_pro.columns:
        df_pro[col] = df_pro[col + '_new']
        df_pro = df_pro.drop(columns=[col + '_new'])

# 填充默认缺失值为 Unknown (以便 04 绘图时可以显示分类)
fill_cols = ['Display Mode', 'V-Sync', 'NVIDIA Reflex Low Latency', 'NVIDIA G-Sync']
df_pro[fill_cols] = df_pro[fill_cols].fillna('Unknown')

df_pro.to_csv('data/cs2_pro_2026_Active_Final.csv', index=False, encoding='utf-8-sig')
print(f'✅ 特征工程补丁完成: 现有 {len(df_pro.columns)} 个维度')

In [ ]:
# Cell 5: K-Means 聚类分析 (无监督学习)
# 目的: 基于 eDPI、分辨率、刷新率、准星间隙和雷达缩放 5 个核心参数，为选手进行角色聚类 (五大流派分类)
# 输入: data/cs2_pro_2026_Active_Final.csv
# 输出: 聚类结果及聚类代表选手
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']

df_pro = pd.read_csv('data/cs2_pro_2026_Active_Final.csv')
features = ['eDPI', 'Res_Width', 'Hz', 'Gap', 'Radar Map Zoom']
X = df_pro[features + ['Player']].dropna().copy()

# 1. 均值归一化 (防止 eDPI 权重过大)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X[features])

# 2. 训练 KMeans 聚类
kmeans = KMeans(n_clusters=5, random_state=7800, n_init=10)
X['Cluster'] = kmeans.fit_predict(X_scaled)

# 3. 计算每个选手距所属中心点的曼哈顿距离 (寻找代表选手)
distances = kmeans.transform(X_scaled)
X['Distance_to_Center'] = [distances[i, c] for i, c in enumerate(X['Cluster'])]

print('🎯 K-Means 聚类完成 (5组选手)')
for i in range(5):
    reps = X[X['Cluster']==i].nsmallest(3, 'Distance_to_Center')['Player'].tolist()
    print(f'  Cluster {i} 代表选手: {", ".join(reps)}')

In [ ]:
# Cell 6: 相关性深度透视
# 目的: 探测硬件配置与设置参数之间的线性相关性（如 eDPI 与分辨率的强关联）
# 输入: data/cs2_pro_2026_Active_Final.csv
# 输出: 皮尔逊相关系数热力图
numeric_df = df_pro.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('设置参数相关性矩阵 (Pearson Correlation)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n🔗 强关联发现 (Top 5):')
pairs = corr.unstack().reset_index()
pairs = pairs[pairs['level_0'] < pairs['level_1']] # 过滤重复对
pairs['abs'] = pairs[0].abs()
top = pairs.nlargest(5, 'abs')
for _, row in top.iterrows():
    print(f"  {row['level_0']} ↔ {row['level_1']}: 系数为 {row[0]:.3f}")

In [ ]:
# Cell 7: 分析就绪检查
# 目的: 最终检查数据集规模、维度和完整性，确保下游可视化无误
# 输出: 就绪报告
df_pro = pd.read_csv('data/cs2_pro_2026_Active_Final.csv')
print(f'✅ 全面核对完毕: 现有 {len(df_pro)} 名选手，维度 {len(df_pro.columns)} 个')

# 统计核心指标缺失情况
missing = df_pro.isnull().sum()
if missing.sum() > 0:
    for col, cnt in missing.nlargest(3).items():
        if cnt > 0: print(f'  ⚠️  注意 {col}: 缺失 {cnt} (正常)')
else:
    print('✅ 全字段数据完整')
print('\n✅ 统计分析阶段所有 pipeline 流程结束。Ready for final report presentation.')